In [1]:
# Creating a dataPipeline

In [2]:
import cv2
import tensorflow as tf
import numpy as np
from mtcnn import MTCNN
from typing import List

detector = MTCNN()

def extract_mouth(points, frame_shape, margin = 10):

    left = points[0]['keypoints']['mouth_left']
    right = points[0]['keypoints']['mouth_right']
    top = points[0]['keypoints']['nose']
    x1 = max(left[0] - margin, 0)
    y1 = max(top[1]+10, 0)
    x2 = min(right[0] + margin, frame_shape[1])
    y2 = min(right[1] + margin*2 , frame_shape[0])

    return y1, y2, x1, x2

In [3]:
def load_video(path: str, resize) -> List[float]: 

    frames = []
    cap = cv2.VideoCapture(path)

    ret, frame = cap.read()

    results = detector.detect_faces(frame)

    y1, y2, x1, x2 = extract_mouth(results, frame.shape)
    cap.release()
    cap = cv2.VideoCapture(path)

    for _ in range(int(cap.get(cv2.CAP_PROP_FRAME_COUNT))):

        ret, frame = cap.read()

        
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        gray = gray[y1:y2, x1:x2]
        gray =  np.expand_dims(gray, axis=-1) 
        gray_resized = tf.image.resize(gray, resize)
        frames.append(gray_resized)
    cap.release()

    # Convert to tensor and normalize
    frames = tf.stack(frames)
    mean = tf.math.reduce_mean(frames)
    std = tf.math.reduce_std(tf.cast(frames, tf.float32))

    return tf.cast((frames - mean), tf.float32) / std
    
    

In [76]:
file_name = "bbaf1p"
speaker = "s4_processed"

In [77]:
video = f"..\\testVideos\\{speaker}\\{file_name}.mpg".format(speaker,file_name)

In [78]:
align_path =  f"..\\testVideos\\{speaker}\\align\\{file_name}.align".format(speaker,file_name)

In [87]:
frames = load_video(video, resize=(48,96))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step


IndexError: list index out of range

In [88]:
frames

<tf.Tensor: shape=(75, 48, 96, 1), dtype=float32, numpy=
array([[[[ 1.4700634 ],
         [ 1.425786  ],
         [ 1.353332  ],
         ...,
         [ 0.6267792 ],
         [ 0.59055215],
         [ 0.56841344]],

        [[ 1.4378617 ],
         [ 1.4046537 ],
         [ 1.3503131 ],
         ...,
         [ 0.5945774 ],
         [ 0.5583504 ],
         [ 0.53621167]],

        [[ 1.3949257 ],
         [ 1.372787  ],
         [ 1.3365599 ],
         ...,
         [ 0.5623756 ],
         [ 0.5261486 ],
         [ 0.50400984]],

        ...,

        [[ 0.6757521 ],
         [ 0.6499242 ],
         [ 0.6076594 ],
         ...,
         [-1.3482641 ],
         [-1.3844912 ],
         [-1.4066299 ]],

        [[ 0.4074045 ],
         [ 0.39633515],
         [ 0.37822163],
         ...,
         [-1.4663371 ],
         [-1.5025641 ],
         [-1.5247028 ]],

        [[ 0.181992  ],
         [ 0.181992  ],
         [ 0.181992  ],
         ...,
         [-1.5629424 ],
         [-1.599169

In [80]:
import string

In [81]:
vocab = string.ascii_lowercase + " "
vocab = list(vocab)

In [82]:
char_to_num = tf.keras.layers.StringLookup(vocabulary=vocab, oov_token="")
num_to_char = tf.keras.layers.StringLookup(vocabulary=char_to_num.get_vocabulary(), oov_token="", invert=True)

In [67]:
def load_alignment(path : str):
    # path = bytes.decode(path.numpy())
    with open(path, "r") as f:
        lines = f.readlines()
    tokens = []
    for line in lines:
        start, end, text = line.split()
        if text!='sil':
            tokens.append(text)
    chars = list(" ".join(tokens))
    return char_to_num(chars)

In [83]:
actual = load_alignment(align_path)

In [84]:
input_frames = np.array([frames.numpy()])

In [70]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Conv3D, Dense, LSTM, Bidirectional, Dropout, 
                                     MaxPool3D, Activation, Reshape, SpatialDropout3D, 
                                     BatchNormalization, TimeDistributed, Flatten, Input)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import LearningRateScheduler, ModelCheckpoint
import tensorflow as tf
from tensorflow.keras import layers, models

In [71]:
inf_model = Sequential()
inf_model.add(Conv3D(128, 3, input_shape=(75,48,96,1), padding='same'))
inf_model.add(Activation('relu'))
inf_model.add(MaxPool3D((1,2,2)))

inf_model.add(Conv3D(128, 3, padding='same'))
inf_model.add(Activation('relu'))
inf_model.add(MaxPool3D((1,2,2)))

inf_model.add(Conv3D(75, 3, padding='same'))
inf_model.add(Activation('relu'))
inf_model.add(MaxPool3D((1,2,2)))

inf_model.add(TimeDistributed(Reshape((-1,))))  # flatten manually


inf_model.add(Bidirectional(LSTM(256, kernel_initializer='Orthogonal', return_sequences=True)))
inf_model.add(Dropout(.5))

inf_model.add(Bidirectional(LSTM(256, kernel_initializer='Orthogonal', return_sequences=True)))
inf_model.add(Dropout(.5))

inf_model.add(Dense(char_to_num.vocabulary_size()+1, kernel_initializer='he_normal', activation='softmax'))

C:\Users\manik\Desktop\Capstone_Final_Project\lipreading_env\lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [72]:
inf_model.load_weights("..\\models\\checkpoint.weights.h5")

In [85]:
yhat = inf_model.predict(input_frames)
decoded = tf.keras.backend.ctc_decode(yhat, [75], greedy=False)[0][0].numpy()

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 625ms/step


In [86]:
print('Original:', tf.strings.reduce_join(num_to_char(actual)).numpy().decode('utf-8'))
print('Prediction:', tf.strings.reduce_join(num_to_char(decoded[0])).numpy().decode('utf-8'))

Original: bin blue at f one please
Prediction: lay blue a nine again
